In [ ]:
import oxpy
import multiprocessing
# Running oxpy multiple times from the same jupyter kernel crashes the kernel for some reason
# So we will be starting simulations from a separate thread.
def spawn(f, kwargs=None):
    if kwargs == None:
        kwargs = {}
    p = multiprocessing.Process(target=f, kwargs = kwargs)
    p.start()
    return p

# Sets up and runs the simulation
def run(**kwargs):
    with oxpy.Context():
        inp = oxpy.InputFile()
        inp.init_from_filename("input")

        # Modify inputfile at runtime
        # all input parameters provided to oxDNA need to be strings 
        for k,v in kwargs.items(): 
            inp[k] = v
        
        manager = oxpy.OxpyManager(inp)
        manager.run_complete()

p = None

In [ ]:
from oxDNA_analysis_tools.mean import mean
from oxDNA_analysis_tools.deviations import deviations
from oxDNA_analysis_tools.UTILS.RyeReader import describe

top = "K1.pdb.top"
traj = "MD_trajectory.dat"
top_info, traj_info = describe(top, traj)

# Compute the mean structure and RMSFs
mean_conf = mean(traj_info, top_info, ncpus=4)
RMSDs, RMSFs = deviations(traj_info, top_info, mean_conf, ncpus=4)

#They come out as numpy arrays, need to be a dict with a list for visualization
RMSFs = {"RMSF": RMSFs.tolist()}

In [ ]:
from oxDNA_analysis_tools.UTILS.oxview import oxdna_conf

# Display python objects in an oxview iframe
oxdna_conf(top_info, mean_conf, RMSFs)